In [200]:
# ipca_pipeline.py (or paste into a notebook cell)

from __future__ import annotations
import os
import contextlib
from dataclasses import dataclass
from typing import List, Tuple, Optional, Dict, Any

import numpy as np
import pandas as pd
from ipca import InstrumentedPCA

def run_pipeline(
    start_yyyymm: str = "199001",
    end_yyyymm: str = "202412",
    max_nan_frac: float = 0.6,
    cutoff: str = "2015-12-31",
    n_factors: int = 3,
) -> Dict[str, Any]:
    df_raw = download_openap_data(start_yyyymm, end_yyyymm)
    df_keep, dropped = remove_mostly_nan_columns(df_raw, max_nan_frac=max_nan_frac)
    df_filled = fill_remaining_missing(df_keep)

    panel = build_ipca_panel(df_filled, ret_col="excess_ret", shift_target=True)
    train_df, test_df = split_train_test_by_date(panel.df, cutoff=cutoff)

    X_train = train_df[panel.char_cols].to_numpy(np.float64)
    y_train = train_df["y_ipca"].to_numpy(np.float64)
    idx_train = train_df[["i_idx", "t_idx"]].to_numpy(np.int64)

    X_test = test_df[panel.char_cols].to_numpy(np.float64)
    y_test = test_df["y_ipca"].to_numpy(np.float64)
    idx_test = test_df[["i_idx", "t_idx"]].to_numpy(np.int64)

    mod = fit_ipca(X_train, y_train, idx_train, n_factors=n_factors, silent=True)
    r2_train = score_ipca(mod, X_train, y_train, idx_train, mean_factor=False)
    r2_test = score_ipca(mod, X_test, y_test, idx_test, mean_factor=True)  # unseen dates

    return {
        "model": mod,
        "dropped_cols": dropped,
        "char_cols": panel.char_cols,
        "r2_train": r2_train,
        "r2_test_mean_factor": r2_test,
        "train_shape": X_train.shape,
        "test_shape": X_test.shape,
    }


In [205]:
%cd ..
from src.ipca_workflow import IPCAWorkflow

/Users/apple/Desktop/Academics/Research_work/TheVirtueOfComplexity_PaperReplication_Experimentation-master


In [ ]:
new_mod = IPCAWorkflow()
new_mod.download_openap_data(start_yyyymm="199001", end_yyyymm="202601")

WRDS recommends setting up a .pgpass file.
You can create this file yourself at any time with the create_pgpass_file() function.
Loading library list...
Done
